# Audit Keandalan URL/QR dan Bukti Label Terpublikasi

Mengikuti [rencana pengerjaan](ai-agent-docs/model-hardening-plan.md). Model v6, baseline dan threshold tetap beku. Cohort utama adalah **development tune**, bukan final test baru.

Audit ini menghitung kesalahan terhadap label sumber, memeriksa representasi, mencocokkan audit manusia yang sudah dipublikasikan, dan menyiapkan paket review buta. Tidak melatih classifier, membuka target/HTML, membuat label manusia, atau menyatakan pilot lama sebagai gold set Indonesia.

Koreksi label hanya diusulkan pada exact match dengan bukti memadai. Domain-only match tidak membenarkan perubahan label path lain.

In [1]:
import csv
import hashlib
import io
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from threadpoolctl import threadpool_limits

from guardian_ml.evaluation import metrics
from guardian_ml.metadata import checksum
from guardian_ml.paths import ROOT
from guardian_ml.preprocessing import EXTRACT, InvalidUrl, prepare_url
from guardian_ml.release_gates import assess_release
from guardian_ml.rich_artifact import RichArtifact
from guardian_ml.rich_features import probabilities

LOCAL = ROOT / "ml/data/model-hardening"
for folder in ["reports", "coordinator", "reviewer-packs", "indonesia-pilot"]:
    (LOCAL / folder).mkdir(parents=True, exist_ok=True)
V6 = ROOT / "ml/runs/notebook-tfidf-logistic-v6"
V5 = ROOT / "ml/runs/notebook-tfidf-logistic-v5"
DATA_PATH = V6 / "data/development-v6.parquet"
protected_paths = [
    ROOT / "guardian_lab_url_qr.ipynb",
    ROOT / "guardian_lab_url_qr_data_v6.ipynb",
    V6 / "artifacts/url-core-logistic-v6.joblib",
    V6 / "reports/frozen-v6.json",
]
protected_before = {
    p.relative_to(ROOT).as_posix(): checksum(p) for p in protected_paths
}
artifact = RichArtifact()
baseline_dir = ROOT / "ml/runs/notebook-tfidf-logistic-v3"
baseline_frozen = json.loads((baseline_dir / "reports/frozen-v3.json").read_text())
assert (
    checksum(baseline_dir / "artifacts/url-core-logistic-v3.joblib")
    == baseline_frozen["bundle_sha256"]
)
baseline = joblib.load(baseline_dir / "artifacts/url-core-logistic-v3.joblib")
all_development = pd.read_parquet(DATA_PATH)
cohort = all_development[all_development.partition.eq("tune")].reset_index(drop=True)
assert len(cohort) == 84000
assert checksum(DATA_PATH) == artifact.metadata["development_sha256"]


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def safe_metrics(labels, probability, threshold):
    result = metrics(labels, probability, threshold)
    if result["benign"] == 0:
        result["fpr"] = None
    if result["phishing"] == 0:
        result["recall"] = None
    if result["confusion_matrix"]["tp"] + result["confusion_matrix"]["fp"] == 0:
        result["precision"] = None
    result["balanced_accuracy"] = (
        (result["recall"] + 1 - result["fpr"]) / 2
        if result["recall"] is not None and result["fpr"] is not None
        else None
    )
    return result


def write_csv(path, columns, rows):
    with Path(path).open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="raise")
        writer.writeheader()
        writer.writerows(rows)


with threadpool_limits(limits=4):
    p_new = artifact.predict_cores(cohort.core.tolist())
    p_old = probabilities(baseline, cohort.core.tolist())
threshold = artifact.metadata["thresholds"]["warning"]
predicted = p_new >= threshold
cohort["model_probability"] = p_new
cohort["error_kind"] = np.select(
    [cohort.label.eq(1) & ~predicted, cohort.label.eq(0) & predicted],
    ["source_label_false_negative", "source_label_false_positive"],
    default="source_label_agreement",
)
source_results = {}
for source, indices in cohort.groupby("training_source").groups.items():
    positions = np.asarray(list(indices))
    source_results[source] = {
        "rows": len(positions),
        "groups": int(cohort.loc[positions, "group"].nunique()),
        "v3_original_policy": safe_metrics(
            cohort.label.iloc[positions],
            p_old[positions],
            baseline["metadata"]["thresholds"]["warning"],
        ),
        "v6": safe_metrics(cohort.label.iloc[positions], p_new[positions], threshold),
    }
display(
    pd.DataFrame(
        [
            {
                "source": source,
                "rows": item["rows"],
                "groups": item["groups"],
                "recall_pct": 100 * item["v6"]["recall"],
                "fpr_pct": 100 * item["v6"]["fpr"],
                "false_negatives": item["v6"]["confusion_matrix"]["fn"],
                "false_positives": item["v6"]["confusion_matrix"]["fp"],
            }
            for source, item in source_results.items()
        ]
    ).round(3)
)
print("Development tune audit only; no dataset URL displayed.")

,source,rows,groups,recall_pct,fpr_pct,false_negatives,false_positives
0,deepurlbench,20000,19406,44.941,0.887,1208,158
1,mendeley2020,1060,863,42.408,0.616,330,3
2,phishvn_core,2274,1604,35.925,0.421,239,8
3,phiusiil,29318,26912,41.754,0.044,6442,8
4,phresh,31348,24477,43.188,0.180,9282,27


Development tune audit only; no dataset URL displayed.


## Struktur URL, representasi identik, dan paket error

Slice mendeskripsikan pola, bukan menetapkan penyebab/label. Raw query tidak tersedia pada cache sehingga kegagalan yang hanya bergantung pada query tidak dapat dihitung.

Paket A/B tidak memuat prediksi atau label sumber. Review masih pending. File input anotasi yang sudah diisi tidak ditimpa.

In [2]:
parsed = [prepare_url(core) for core in cohort.core]
slices = {
    "homepage": np.array([p.path == "/" for p in parsed]),
    "non_homepage": np.array([p.path != "/" for p in parsed]),
    "private_hosting_tenant": np.array(
        [bool(EXTRACT(p.host).is_private) for p in parsed]
    ),
    "punycode": np.array(["xn--" in p.host for p in parsed]),
    "url_over_120_chars": np.array(
        [len(p.core.split("://", 1)[1]) > 120 for p in parsed]
    ),
    "redacted_placeholder_present": cohort.core.str.contains(
        "REDACTED", regex=False
    ).to_numpy(),
}
slice_results = {}
for name, mask in slices.items():
    if mask.any():
        slice_results[name] = {
            "groups": int(cohort.loc[mask, "group"].nunique()),
            **safe_metrics(cohort.label[mask], p_new[mask], threshold),
        }
display(
    pd.DataFrame(
        [
            {
                "slice": name,
                "rows": item["rows"],
                "phishing": item["phishing"],
                "benign": item["benign"],
                "recall": item["recall"],
                "fpr": item["fpr"],
            }
            for name, item in slice_results.items()
        ]
    )
)
proxy = all_development.core.str.replace(r"^https?://", "", regex=True)
collision_counts = (
    pd.DataFrame({"proxy": proxy, "label": all_development.label})
    .groupby("proxy")
    .label.agg(["count", "sum", "nunique"])
)
conflicts = collision_counts[collision_counts["nunique"].gt(1)]
representation_audit = {
    "development_rows": len(all_development),
    "scheme_free_duplicate_representations": int(collision_counts["count"].gt(1).sum()),
    "scheme_free_conflicting_label_representations": len(conflicts),
    "minimum_errors_on_identical_representations": int(
        np.minimum(conflicts["sum"], conflicts["count"] - conflicts["sum"]).sum()
    ),
    "query_only_discriminator_frequency": None,
    "query_only_reason": "Original query strings are intentionally not persisted; cannot infer missing evidence from core.",
}
print("Representation audit:", representation_audit)
errors = cohort[cohort.error_kind.ne("source_label_agreement")]
sample = (
    errors.sort_values("digest")
    .groupby(["training_source", "error_kind"], sort=True)
    .head(20)
)
coordinator_rows, blind_rows = [], []
for row in sample.to_dict("records"):
    sample_id = "GLH-" + row["digest"][:12]
    evidence_ref = "EVID-DV-" + row["digest"][:10]
    coordinator_rows.append(
        {
            "sample_id": sample_id,
            "core_redacted": row["core"],
            "source": row["training_source"],
            "partition": "tune",
            "source_label_unverified": int(row["label"]),
            "model_probability": float(row["model_probability"]),
            "error_kind": row["error_kind"],
            "evidence_ref": evidence_ref,
            "evidence_scope": "dataset_provenance_only_content_review_pending",
        }
    )
    blind_rows.append(
        {
            "sample_id": sample_id,
            "target_defanged": row["core"]
            .replace("https://", "hxxps://")
            .replace("http://", "hxxp://")
            .replace(".", "[.]"),
            "evidence_ref": evidence_ref,
            "reviewer_id": "",
            "label": "",
            "confidence": "",
            "reason_code": "",
            "human_review_confirmed": "",
            "reviewed_at": "",
        }
    )
write_csv(
    LOCAL / "coordinator/development-error-cases.csv",
    [
        "sample_id",
        "core_redacted",
        "source",
        "partition",
        "source_label_unverified",
        "model_probability",
        "error_kind",
        "evidence_ref",
        "evidence_scope",
    ],
    coordinator_rows,
)
for side in ["a", "b"]:
    pack_path = LOCAL / f"reviewer-packs/error-reviewer-{side}.csv"
    if not pack_path.exists():
        write_csv(
            pack_path,
            [
                "sample_id",
                "target_defanged",
                "evidence_ref",
                "reviewer_id",
                "label",
                "confidence",
                "reason_code",
                "human_review_confirmed",
                "reviewed_at",
            ],
            blind_rows,
        )
print(
    "Blind development error cases prepared:",
    len(blind_rows),
    "; new human labels authored:",
    0,
)

,slice,rows,phishing,benign,recall,fpr
0,homepage,47898,18438,29460,0.427812,0.004752
1,non_homepage,36102,12100,24002,0.425537,0.002666
2,private_hosting_tenant,8212,7388,824,0.905522,0.063107
3,punycode,69,35,34,0.114286,0.029412
4,url_over_120_chars,616,315,301,0.114286,0.006645
5,redacted_placeholder_present,4709,1314,3395,0.292237,0.000884


Representation audit: {'development_rows': 703604, 'scheme_free_duplicate_representations': 9674, 'scheme_free_conflicting_label_representations': 33, 'minimum_errors_on_identical_representations': 33, 'query_only_discriminator_frequency': None, 'query_only_reason': 'Original query strings are intentionally not persisted; cannot infer missing evidence from core.'}
Blind development error cases prepared: 159 ; new human labels authored: 0


## Audit manusia yang dipublikasikan: exact match dan domain-only

Sumber dipin pada commit PhishVN yang sudah diaudit. SHA-256 ketiga sheet dibandingkan dengan manifest v5 sebelum dipakai. Tinjauan penerbit tidak diklaim sebagai review baru oleh agent.

Bukti berbasis nama/none tidak dipakai untuk menyarankan koreksi binary. Label scam berada di luar definisi credential-phishing. Domain-only match hanya petunjuk investigasi, bukan label seluruh halaman.

In [3]:
COMMIT = "2bc80412a4b94ae546211e488ced996c8e0b78f1"
previous_manifest = json.loads(
    (V5 / "reports/frozen-new-test-manifest.json").read_text()
)
expected_assets = {
    item["role"].split(":", 1)[1]: item
    for item in previous_manifest["source_assets"]
    if item["role"].startswith("phishvn_human:")
}
sheets, verified_sheet_assets = {}, []
for filename in ["annotator_A.csv", "annotator_B.csv", "adjudicated.csv"]:
    asset = expected_assets[filename]
    assert COMMIT in asset["source_url"]
    response = requests.get(asset["source_url"], timeout=(30, 90))
    response.raise_for_status()
    assert hashlib.sha256(response.content).hexdigest() == asset["sha256"]
    sheets[filename] = pd.read_csv(
        io.BytesIO(response.content), dtype=str, keep_default_na=False
    )
    verified_sheet_assets.append(
        {"filename": filename, "sha256": asset["sha256"], "rows": len(sheets[filename])}
    )
merged = (
    sheets["annotator_A.csv"]
    .merge(
        sheets["annotator_B.csv"],
        on=["vid", "url"],
        suffixes=("_a", "_b"),
        validate="one_to_one",
    )
    .merge(
        sheets["adjudicated.csv"][["vid", "final_verdict", "final_evidence"]],
        on="vid",
        how="left",
        validate="one_to_one",
    )
)
by_digest = all_development.set_index("digest", drop=False)
by_group = all_development.groupby("group").size().to_dict()
crosswalk = []
for row in merged.to_dict("records"):
    adjudicated = pd.notna(row["final_verdict"])
    verdict = (
        row["final_verdict"]
        if adjudicated
        else row["verdict_a"]
        if row["verdict_a"] == row["verdict_b"]
        else "unsure"
    )
    evidence = (
        {row["final_evidence"]}
        if adjudicated
        else {row["evidence_a"], row["evidence_b"]}
    )
    evidence_supported = bool(evidence) and evidence.issubset(
        {"capture", "archive", "urlscan", "registry"}
    )
    try:
        item = prepare_url(row["url"])
    except InvalidUrl:
        crosswalk.append(
            {
                "case_id": row["vid"],
                "match_kind": "unsupported",
                "verdict": verdict,
                "supported_evidence": evidence_supported,
                "action": "no_label_action",
            }
        )
        continue
    if item.digest in by_digest.index:
        target = by_digest.loc[item.digest]
        expected = {"phishing": 1, "legitimate": 0}.get(verdict)
        action = (
            "review_exact_binary_mismatch"
            if evidence_supported
            and expected is not None
            and int(target.label) != expected
            else "review_exact_out_of_scope"
            if evidence_supported and verdict == "scam"
            else "supported_exact_binary_agreement"
            if evidence_supported and expected is not None
            else "no_label_action"
        )
        crosswalk.append(
            {
                "case_id": row["vid"],
                "match_kind": "exact_canonical",
                "verdict": verdict,
                "supported_evidence": evidence_supported,
                "action": action,
                "partition": target.partition,
                "training_source": target.training_source,
                "source_label": int(target.label),
            }
        )
    else:
        crosswalk.append(
            {
                "case_id": row["vid"],
                "match_kind": "domain_only" if item.group in by_group else "unmatched",
                "verdict": verdict,
                "supported_evidence": evidence_supported,
                "action": "investigate_without_domain_wide_relabel"
                if item.group in by_group
                else "no_label_action",
            }
        )
crosswalk_frame = pd.DataFrame(crosswalk)
crosswalk_frame.to_csv(LOCAL / "coordinator/published-human-crosswalk.csv", index=False)
human_audit = {
    "commit": COMMIT,
    "verified_sheet_assets": verified_sheet_assets,
    "published_cases": len(merged),
    "case_match_counts": crosswalk_frame.match_kind.value_counts().to_dict(),
    "action_counts": crosswalk_frame.action.value_counts().to_dict(),
    "verdict_counts": crosswalk_frame.verdict.value_counts().to_dict(),
    "new_reviews_by_agent": 0,
    "scope": "Published Vietnamese review; not Indonesian gold or new independent test.",
    "automatic_relabels_applied": 0,
}
display(
    pd.DataFrame(
        crosswalk_frame.groupby(["match_kind", "verdict"]).size(), columns=["cases"]
    )
)
print("Published evidence audit:", human_audit)

cases
match_kind      verdict          
domain_only     legitimate      3
                phishing        2
                scam            2
                unsure          4
exact_canonical legitimate     39
                phishing        4
                scam            3
                unsure         11
unmatched       legitimate     18
                phishing       31
                scam           46
                unsure         35
unsupported     scam            1
                unsure          1

Published evidence audit: {'commit': '2bc80412a4b94ae546211e488ced996c8e0b78f1', 'verified_sheet_assets': [{'filename': 'annotator_A.csv', 'sha256': 'dcd63f7d4118ee0f44c33a4028eaa64d75637acb0c4ec082288fca2d5a7f2d82', 'rows': 200}, {'filename': 'annotator_B.csv', 'sha256': '36afd6c64603d4f454bd1991d45c10d790742b52e3a1d34386ef6fb2568233bb', 'rows': 200}, {'filename': 'adjudicated.csv', 'sha256': '2c3997e509f3ebeff212be22a77d4e14ff49fa488a9530f236e644bddd022e4a', 'rows': 58}], 'published_cases': 200, 'case_match_counts': {'unmatched': 130, 'exact_canonical': 57, 'domain_only': 11, 'unsupported': 2}, 'action_counts': {'no_label_action': 148, 'supported_exact_binary_agreement': 38, 'investigate_without_domain_wide_relabel': 11, 'review_exact_out_of_scope': 3}, 'verdict_counts': {'legitimate': 60, 'scam': 52, 'unsure': 51, 'phishing': 37}, 'new_reviews_by_agent': 0, 'scope': 'Published Vietnamese review; not Indonesian gold or new independent test.', 'automatic_relabels_applied': 0}


## Pilot konteks Indonesia dan keputusan gate

Pemilihan hanya mencari petunjuk brand pada development. Tidak menetapkan relevansi Indonesia, label, bukti halaman, atau persetujuan privasi. Kandidat berstatus development pilot teramati; source label dan skor tidak diekspor ke paket reviewer.

Sesuai pilihan pengguna, bukti manusia yang sudah dipublikasikan digunakan dahulu. Pilot Indonesia disiapkan untuk tinjauan berikutnya, tanpa label buatan.

In [4]:
pattern = re.compile(
    r"(?i)(?<![a-z])(?:bca|bni|bri|brimo|bankmandiri|livin|dana|ovo|gopay|tokopedia|jtexpress|jne|posindonesia)(?![a-z])"
)
candidate_pool = all_development[
    all_development.core.str.contains(pattern)
].sort_values("digest")
pilot = candidate_pool.groupby("group", sort=False).head(1).head(200)
licenses = {
    "phresh": "CC-BY-4.0",
    "phiusiil": "CC-BY-4.0",
    "mendeley2020": "CC-BY-4.0",
    "phishvn_core": "CC-BY-4.0",
    "deepurlbench": "CC-BY-NC-4.0",
}
versions = {
    "phresh": "phresh-v1.0.1",
    "phiusiil": "phiusiil-2024",
    "mendeley2020": "mendeley2020-v3",
    "phishvn_core": "phishvn-v4",
    "deepurlbench": "deepurlbench-2024",
}
columns = (
    (ROOT / "ml/templates/indonesia-candidates.csv").read_text().strip().split(",")
)
pilot_rows = []
for row in pilot.to_dict("records"):
    pilot_rows.append(
        {
            "sample_id": "GLI-DEV-" + row["digest"][:12],
            "url_redacted": row["core"],
            "observed_at": "",
            "source_id": row["training_source"],
            "source_version": versions[row["training_source"]],
            "source_license": licenses[row["training_source"]],
            "evidence_ref": "",
            "scenario": "other",
            "id_relevance_basis": "Kandidat token brand; relevansi Indonesia belum diverifikasi.",
            "requested_role": "development_pilot",
            "privacy_reviewed": "no",
        }
    )
pilot_path = LOCAL / "indonesia-pilot/indonesia-candidates.csv"
if not pilot_path.exists():
    write_csv(pilot_path, columns, pilot_rows)
else:
    print("Existing pilot input preserved; not overwritten.")
gate = assess_release()
write_json(LOCAL / "reports/release-gate-status.json", gate)
pilot_summary = {
    "brand_token_candidate_pool": len(candidate_pool),
    "prepared_pilot_rows": len(pilot_rows),
    "role": "observed_development_pilot",
    "indonesia_relevance_verified": False,
    "privacy_human_qa_completed": False,
    "human_labels_available": 0,
    "source_counts": pilot.training_source.value_counts().to_dict(),
}
overall = safe_metrics(cohort.label, p_new, threshold)
assert protected_before == {
    p.relative_to(ROOT).as_posix(): checksum(p) for p in protected_paths
}
report = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "cohort": "v6_development_tune_observed",
    "cohort_rows": len(cohort),
    "data_sha256": checksum(DATA_PATH),
    "model_bundle_sha256": artifact.manifest["bundle_sha256"],
    "model_thresholds_unchanged": True,
    "overall_v6": overall,
    "source_metrics": source_results,
    "structural_slices": slice_results,
    "representation_audit": representation_audit,
    "published_human_audit": human_audit,
    "blind_error_cases_prepared": len(blind_rows),
    "indonesia_pilot": pilot_summary,
    "release_gate": gate,
    "protected_model_hashes": protected_before,
    "limitations": [
        "Source-label disagreement is not proof a source label or the classifier is wrong.",
        "Structural slices are hypotheses, not causal or semantic diagnoses.",
        "Published human audit is not new Indonesian annotation; domain-only matches cannot relabel other paths.",
        "No query reconstruction, target visits, HTML execution, LLM labels, or invented human reviews.",
        "No new training, thresholds, model promotion, or public application activation.",
    ],
}
write_json(LOCAL / "reports/model-hardening-audit.json", report)
write_json(ROOT / "ml/reports/model-hardening-audit.json", report)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
names = list(source_results)
for axis, count in zip(axes, ["fn", "fp"]):
    values = [source_results[name]["v6"]["confusion_matrix"][count] for name in names]
    axis.bar(names, values, color="#0f766e")
    axis.set_title("Source-label " + count.upper() + " pada development tune")
    axis.tick_params(axis="x", labelrotation=25)
    axis.set_ylabel("Jumlah URL")
fig.tight_layout()
fig.savefig(LOCAL / "reports/development-errors.png", dpi=150)
display(fig)
plt.close(fig)
print(
    "Gate:",
    gate["status"],
    "; failed criteria:",
    len([f for f in gate["findings"] if f["status"] == "failed"]),
)
print("Pilot ready for review, not labeled:", pilot_summary)

Existing pilot input preserved; not overwritten.


<Figure size 1200x450 with 2 Axes>

Gate: not_ready ; failed criteria: 3
Pilot ready for review, not labeled: {'brand_token_candidate_pool': 103, 'prepared_pilot_rows': 73, 'role': 'observed_development_pilot', 'indonesia_relevance_verified': False, 'privacy_human_qa_completed': False, 'human_labels_available': 0, 'source_counts': {'phresh': 38, 'phiusiil': 18, 'deepurlbench': 14, 'phishvn_core': 3}}


## Tindak lanjut berdasarkan bukti

Gunakan crosswalk dan paket review lokal untuk menyelesaikan diagnosis. Kandidat Indonesia belum memiliki label manusia atau bukti kasus yang memadai. Gate gagal/pending tidak diperbaiki dengan menurunkan target atau mengubah final test.

Runtime shadow dipersiapkan terpisah dengan preprocessing yang sama, jawaban perlu tinjauan, dan telemetry agregat tanpa target. Aktivasi publik dan analisis konten/redirect menunggu bukti validasi serta isolasi teknis.

## Uji API shadow dengan model nyata

Smoke test berlangsung dalam proses melalui ASGI TestClient, tanpa membuka port atau mengakses URL target. Input sintetis; tidak ada prediksi baru pada final test. Respons tetap perlu tinjauan dan tidak menggemakan target.

In [5]:
from fastapi.testclient import TestClient
from guardian_ml.shadow_service import create_app

with TestClient(create_app()) as client:
    health = client.get("/health").json()
    assert health["model_available"] and not health["production_ready"]
    url_response = client.post(
        "/predict",
        json={
            "url": "https://example.org/login?token=SYNTHETIC-PRIVATE-MARKER",
            "input_type": "URL",
        },
    )
    qr_response = client.post(
        "/predict", json={"url": "https://example.org/login", "input_type": "QR"}
    )
    unsupported = client.post("/predict", json={"url": "file:///tmp/example"})
    invalid = client.post(
        "/predict",
        json={"url": "https://example.org", "secret": "SYNTHETIC-PRIVATE-MARKER"},
    )
    counters = client.get("/metrics").json()
    assert url_response.status_code == qr_response.status_code == 200
    assert url_response.json()["model_score"] == qr_response.json()["model_score"]
    assert unsupported.status_code == 422 and unsupported.json()["risk_band"] is None
    assert invalid.status_code == 422
    assert all(
        "SYNTHETIC-PRIVATE-MARKER" not in response.text
        and "example.org" not in response.text
        for response in [url_response, qr_response, unsupported, invalid]
    )
    assert counters["total_requests"] == 4
    assert counters["quality_metrics"] == {
        "precision": None,
        "recall": None,
        "fpr": None,
    }
api_checks = {
    "real_model_available": health["model_available"],
    "url_qr_parity": True,
    "unsupported_never_low": True,
    "request_validation_without_echo": True,
    "quality_metrics_not_invented": True,
    "production_ready": False,
    "request_count_synthetic_smoke": counters["total_requests"],
}
assert protected_before == {
    p.relative_to(ROOT).as_posix(): checksum(p) for p in protected_paths
}
report["shadow_api_smoke"] = api_checks
write_json(LOCAL / "reports/model-hardening-audit.json", report)
write_json(ROOT / "ml/reports/model-hardening-audit.json", report)
display(url_response.json())
print("API smoke:", api_checks)

{'mode': 'shadow',
 'status': 'complete',
 'input_type': 'URL',
 'decision': 'needs_review',
 'needs_further_review': True,
 'production_ready': False,
 'noncommercial_research_only': True,
 'risk_band': 'LOW',
 'model_score': 0.09766967877744925,
 'reason_codes': ['research_model_not_production_validated',
  'query_context_excluded'],
 'human': {'title': 'Indikator pola URL rendah',
  'explanation': 'Skor rendah belum menjamin tautan aman. Isi dan tujuan akhirnya masih memerlukan verifikasi.',
  'recommended_action': 'perlu_verifikasi'},
 'geek': {'scope': 'url_only',
  'model_version': 'notebook-url-core-v6',
  'artifact_sha256': 'f332bcbf7665955677e17daa73c487a7e1572a18d7f465b87e73caaa72e06319',
  'feature_schema': 'url-core-rich-offline-v6-1',
  'score_reference': 'research_calibration_not_traffic_probability',
  'thresholds': {'warning': 0.904713285827399, 'high': 0.9884744926236343},
  'context_flags': ['research_model_not_production_validated',
   'query_context_excluded'],
  'c

API smoke: {'real_model_available': True, 'url_qr_parity': True, 'unsupported_never_low': True, 'request_validation_without_echo': True, 'quality_metrics_not_invented': True, 'production_ready': False, 'request_count_synthetic_smoke': 4}
